# Multiverse Hybrid v3.0 — Stage 2 PRICE / EV 2000 v2

v1の科学計算はPASS済みです。v2は巨大ZIP/再ハッシュ処理を行わず、既存Stage 2成果物のSHAとfirewallだけを検証して緑終了させる回復版です。

**再計算はしません。iPhoneでは『ランタイム → すべてのセルを実行』だけで構いません。**


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'
CAT=OUT/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
QUALITY=OUT/'STAGE2_PRICE_EV_CATALOG_QUALITY_v1.json'
RECEIPT=OUT/'STAGE2_PRICE_EV_RECEIPT_v1.json'
RECOVERY=OUT/'STAGE2_V2_POSTPROCESS_RECOVERY_RECEIPT.json'

EXPECTED={
 'catalog':'34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc',
 'quality':'2f9398ebb7ff188a507da2f5fa627ac6f8dcd4de0643c0f1ef19ceeffe917656',
 'price':'2ca98097f74e5282fdc9c91629083f39bef4dafb94a1fc4f7e510acadefc407b',
 'prob':'6348d9af2a535578cf454afca52ea2c944cb6c50cab87f6e6ffa75149880b526',
}

def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()

for p in (CAT,QUALITY,RECEIPT):
    if not p.is_file(): raise RuntimeError(f'FAIL-CLOSED missing {p}')

r=json.loads(RECEIPT.read_text(encoding='utf-8'))
q=json.loads(QUALITY.read_text(encoding='utf-8'))
cat_sha=sha256(CAT)
quality_sha=sha256(QUALITY)

checks={
 'receipt_status_pass': r.get('status')=='PASS',
 'quality_status_pass': q.get('status')=='PASS',
 'catalog_sha_match': cat_sha==EXPECTED['catalog']==r.get('catalog_sha256')==q.get('output_sha256'),
 'quality_sha_match': quality_sha==EXPECTED['quality']==r.get('quality_sha256'),
 'price_sha_match': r.get('price_sha256')==EXPECTED['price']==q.get('price_sha256'),
 'prob_sha_match': r.get('ticket_probability_sha256')==EXPECTED['prob']==q.get('ticket_probability_sha256'),
 'races_2000': r.get('races')==2000 and q.get('races')==2000,
 'rows_4000': r.get('output_rows')==4000 and q.get('output_rows')==4000,
 'ticket_join_mismatch_zero': r.get('ticket_join_mismatches')==0 and q.get('ticket_join_mismatches')==0,
 'result_access_false': r.get('result_access') is False and q.get('result_access') is False,
 'settlement_access_false': r.get('settlement_access') is False and q.get('settlement_access') is False,
 'realized_roi_false': r.get('realized_roi_computed') is False and q.get('realized_roi_computed') is False,
 'threshold_false': r.get('threshold_selected') is False and q.get('threshold_selected') is False,
 'portfolio_false': r.get('portfolio_constructed') is False and q.get('portfolio_constructed') is False,
 'holdout_sealed': r.get('ECON_HOLDOUT1000')=='SEALED' and q.get('ECON_HOLDOUT1000')=='SEALED',
}
bad=[k for k,v in checks.items() if not v]
if bad: raise RuntimeError('FAIL-CLOSED Stage2 v2 recovery check failed: '+','.join(bad))

recovery={
 'record':'STAGE2_V2_POSTPROCESS_RECOVERY_RECEIPT',
 'status':'PASS_EXISTING_STAGE2_ACCEPTED_NO_RECOMPUTE',
 'scientific_stage2_recomputed':False,
 'catalog_sha256':cat_sha,
 'quality_sha256':quality_sha,
 'races':2000,
 'output_rows':4000,
 'ticket_join_mismatches':0,
 'result_access':False,
 'settlement_access':False,
 'realized_roi_computed':False,
 'threshold_selected':False,
 'portfolio_constructed':False,
 'scientific_trial_count':0,
 'ECON_HOLDOUT1000':'SEALED',
 'v1_large_zip_rehash_required':False,
 'v2_large_zip_created':False,
 'checks':checks,
}
RECOVERY.write_text(json.dumps(recovery,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
print('✅ STAGE 2 PASS — v2 recovery complete; NO RECOMPUTE')
print('Catalog SHA:',cat_sha)
print('Quality SHA:',quality_sha)
print('2000 races / 4000 model-race rows / join mismatch 0')
print('RESULT/PAYOUT/Settlement/realized ROI access = none')
print('Threshold/Portfolio selection = none')
print('ECON_HOLDOUT1000 = SEALED')
